# Run Corrected Experiments on IT Monitor Datasets

This notebook runs causal discovery experiments on IT monitor datasets with the correct understanding:
- Each activity type (Antivirus, Web, MoM) has only one dataset
- **TCDF** is non-deterministic and will be run multiple times with different seeds for averaging
- **DynoTEARS** is deterministic and will be run only once
- Other methods (VarLiNGAM, PCMCI, VCDF) are deterministic and run once
- Records individual runs for TCDF plus averages
- F1 scores are labeled as Summary_F1 to clarify they are summary-level evaluations


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import graphviz as gv
import math

print([np.__version__, pd.__version__])
np.set_printoptions(precision=3, suppress=True)

from src.data_preprocessing import preprocess_data
from src.plotting import plot_heatmap, plot_causal_graph
from src.causal_matrix_evaluation import evaluate_causal_matrices
from src.matrix_utils import read_matrices_from_csv, save_matrices, get_summary_matrix
from src.run_causal_discovery import run_pcmci, run_vcdf_pcmci, run_varlingam, run_vcdf_varlingam, run_tcdf, run_dynotears


In [ ]:
def get_dataset_info(activity_type):
    """
    Get the correct dataset file path for each activity type.
    Based on the description, we use:
    - MoM 2 (longer window) for Middleware_oriented_message_Activity
    - Strategy 2 variants for both Antivirus and Web
    """
    activity_path = f'data/real/IT_monitor/{activity_type}'
    
    if activity_type == 'Middleware_oriented_message_Activity':
        # Use dataset_1.csv as it's likely the MoM 2 (longer window)
        return f'{activity_path}/dataset_1.csv'
    elif activity_type == 'Antivirus_Activity':
        # Use preprocessed_2.csv as it's Strategy 2
        return f'{activity_path}/preprocessed_2.csv'
    elif activity_type == 'Web_Activity':
        # Use preprocessed_2.csv as it's Strategy 2  
        return f'{activity_path}/preprocessed_2.csv'
    else:
        raise ValueError(f"Unknown activity type: {activity_type}")


In [ ]:
def run_experiments(activity_types, methods=['pcmci', 'vcdf_pcmci', 'varlingam', 'vcdf_varlingam', 'tcdf', 'dynotears'], n_runs_nondeterministic=10):
    """
    Run causal discovery experiments on IT monitor datasets.
    
    Args:
        activity_types: List of activity types to process
        methods: List of methods to run
        n_runs_nondeterministic: Number of runs for non-deterministic methods (only TCDF)
    """
    # Define which methods are non-deterministic
    # DynoTEARS is actually deterministic, only TCDF is non-deterministic
    nondeterministic_methods = ['tcdf']
    
    for activity_type in activity_types:
        print(f"\n{'='*60}")
        print(f"Running experiments for {activity_type}...")
        print(f"{'='*60}")
        
        # Load ground truth
        ground_truths_path = f'data/real/IT_monitor/{activity_type}/ground_truth.csv'
        ground_truths = read_matrices_from_csv(ground_truths_path)
        
        if ground_truths is None:
            print(f"Skipping {activity_type} due to missing ground truths")
            continue
            
        ground_truth = ground_truths[0]  # Use first matrix as ground truth
        print(f"Ground truth shape: {ground_truth.shape}")
        print(f"Ground truth non-zero elements: {np.count_nonzero(ground_truth)}")
        
        # Load and preprocess data
        data_path = get_dataset_info(activity_type)
        print(f"Loading data from: {data_path}")
        
        if not os.path.exists(data_path):
            print(f"Data file not found: {data_path}")
            continue
            
        data = pd.read_csv(data_path)
        print(f"Original data shape: {data.shape}")
        
        columns = data.columns.tolist()
        
        # Remove timestamp columns if present
        for time_col in ['Date', 'timestamp', 'Timestamp']:
            if time_col in columns:
                data = data.drop([time_col], axis=1)
                columns.remove(time_col)
                print(f"Removed time column: {time_col}")
        
        data = data.values
        data = preprocess_data(data, columns)
        print(f"Preprocessed data shape: {data.shape}")
        
        # Run each method
        for method in methods:
            print(f"\n{'-'*40}")
            print(f"Running {method} on {activity_type}")
            print(f"{'-'*40}")
            
            # Determine number of runs
            if method in nondeterministic_methods:
                n_runs = n_runs_nondeterministic
                print(f"Non-deterministic method ({method}): running {n_runs} times with different seeds")
            else:
                n_runs = 1
                print(f"Deterministic method ({method}): running once")
            
            try:
                results = []
                all_matrices_for_saving = []
                
                # Run method multiple times if needed
                for run_idx in range(n_runs):
                    if n_runs > 1:
                        print(f"  Run {run_idx + 1}/{n_runs} (seed: {1111 + run_idx})")
                    
                    start_time = time.time()
                    
                    # Run causal discovery method
                    if method == 'pcmci':
                        adjacency_matrices = run_pcmci(data)
                    elif method == 'vcdf_pcmci':
                        adjacency_matrices = run_vcdf_pcmci(data)
                    elif method == 'varlingam':
                        adjacency_matrices = run_varlingam(data)
                    elif method == 'vcdf_varlingam':
                        adjacency_matrices = run_vcdf_varlingam(data)
                    elif method == 'tcdf':
                        # For TCDF, use different seeds for different runs
                        seed = 1111 + run_idx if n_runs > 1 else 1111
                        adjacency_matrices = run_tcdf(data, seed=seed)
                    elif method == 'dynotears':
                        adjacency_matrices = run_dynotears(data)
                    else:
                        raise ValueError(f"Unknown method: {method}")
                    
                    runtime = round(time.time() - start_time, 4)
                    
                    # Store matrices from this run for averaging
                    all_matrices_for_saving.extend(adjacency_matrices)
                    
                    # Get summary matrix from this run
                    summary_matrix = get_summary_matrix(adjacency_matrices)
                    
                    # Evaluate summary matrix against ground truth
                    evaluation = evaluate_causal_matrices([ground_truth], [summary_matrix])
                    
                    # Store results for this run
                    run_result = {
                        'dataset': f'run_{run_idx + 1}' if n_runs > 1 else 'single_run',
                        'SHD': evaluation['shd'],
                        'Summary_F1': evaluation['f1'],  # Explicitly label as Summary F1
                        'Summary_F1_sign': evaluation['f1_sign'],  # Explicitly label as Summary F1_sign
                        'runtime': runtime
                    }
                    results.append(run_result)
                
                # Calculate averages if multiple runs
                if n_runs > 1:
                    averages = {
                        metric: np.mean([r[metric] for r in results if isinstance(r[metric], (int, float))])
                        for metric in ['SHD', 'Summary_F1', 'Summary_F1_sign', 'runtime']
                    }
                    
                    # Add average row
                    results.append({
                        'dataset': 'Average',
                        'SHD': f"{averages['SHD']:.2f}",
                        'Summary_F1': f"{averages['Summary_F1']:.3f}",
                        'Summary_F1_sign': f"{averages['Summary_F1_sign']:.3f}",
                        'runtime': f"{averages['runtime']:.4f}"
                    })
                
                # Create results directory
                results_dir = f'results/real/IT_monitor/{activity_type}'
                os.makedirs(results_dir, exist_ok=True)
                
                # Save results
                df_results = pd.DataFrame(results)
                results_path = f'{results_dir}/performance_{method}.csv'
                df_results.to_csv(results_path, index=False)
                
                # For saving matrices, use the summary from all runs combined
                final_summary_matrix = get_summary_matrix(all_matrices_for_saving)
                
                # Save summary matrix
                summary_path = f'{results_dir}/sum_adj_matrix_{method}.csv'
                save_matrices([final_summary_matrix], summary_path)
                
                # Save full adjacency matrices
                matrices_path = f'{results_dir}/adj_matrices_{method}.csv'
                save_matrices(all_matrices_for_saving, matrices_path)
                
                # Print summary
                if n_runs > 1:
                    print(f"\nResults summary for {method} (averaged over {n_runs} runs):")
                    print(f"  SHD: {averages['SHD']:.2f}")
                    print(f"  Summary F1: {averages['Summary_F1']:.3f}")
                    print(f"  Summary F1_sign: {averages['Summary_F1_sign']:.3f}")
                    print(f"  Runtime (avg): {averages['runtime']:.4f}s")
                else:
                    print(f"\nResults for {method}:")
                    print(f"  SHD: {results[0]['SHD']:.2f}")
                    print(f"  Summary F1: {results[0]['Summary_F1']:.3f}")
                    print(f"  Summary F1_sign: {results[0]['Summary_F1_sign']:.3f}")
                    print(f"  Runtime: {results[0]['runtime']:.4f}s")
                
            except Exception as e:
                print(f"Error running {method} on {activity_type}: {str(e)}")
                continue
    
    print(f"\n{'='*60}")
    print("All experiments completed!")
    print(f"{'='*60}")


In [ ]:
# Define activity types to process
activity_types = [
    'Antivirus_Activity',
    'Middleware_oriented_message_Activity',
    'Web_Activity'
]

# Define methods to run
methods_to_run = ['pcmci', 'vcdf_pcmci', 'varlingam', 'vcdf_varlingam', 'tcdf', 'dynotears']

# Only TCDF is non-deterministic and will be run 10 times with different seeds
run_experiments(activity_types, methods=methods_to_run, n_runs_nondeterministic=10)


In [ ]:
# Optional: Create a summary table of all results
def create_summary_table(activity_types, methods):
    """
    Create a summary table of all experimental results.
    """
    summary_data = []
    
    for activity_type in activity_types:
        for method in methods:
            results_path = f'results/real/IT_monitor/{activity_type}/performance_{method}.csv'
            
            if os.path.exists(results_path):
                df = pd.read_csv(results_path)
                
                # Get the average/final row (last row if multiple runs, first row if single run)
                if len(df) > 1:
                    # Multiple runs - get the average row
                    summary_row = df[df['dataset'] == 'Average']
                    if len(summary_row) > 0:
                        result_dict = summary_row.iloc[0].to_dict()
                        result_dict['activity_type'] = activity_type
                        result_dict['method'] = method
                        summary_data.append(result_dict)
                else:
                    # Single run
                    result_dict = df.iloc[0].to_dict()
                    result_dict['activity_type'] = activity_type
                    result_dict['method'] = method
                    summary_data.append(result_dict)
    
    if summary_data:
        summary_df = pd.DataFrame(summary_data)
        
        # Reorder columns
        col_order = ['activity_type', 'method', 'dataset', 'SHD', 'Summary_F1', 'Summary_F1_sign', 'runtime']
        summary_df = summary_df[col_order]
        
        # Save summary
        summary_path = 'results/real/IT_monitor/summary_all_methods.csv'
        summary_df.to_csv(summary_path, index=False)
        
        print("\nSummary of all results:")
        print(summary_df.to_string(index=False))
        
        return summary_df
    else:
        print("No results found to summarize")
        return None

# Create summary table
summary_df = create_summary_table(activity_types, methods_to_run)
